In [1]:
import os
import csv
from datetime import datetime, timedelta
import requests
from requests.auth import HTTPBasicAuth
import time


In [ ]:
CITY = os.getenv("CITY")
USERNAME = os.getenv("USERNAME")
PASSWORD = os.getenv("PASSWORD")

BASE_URL = f"https://{CITY}.pulse.eco/rest"

OUT_DIR = "pulse_data"
os.makedirs(OUT_DIR, exist_ok=True)

In [30]:
def get_sensors():
    url = f"{BASE_URL}/sensor"
    r = requests.get(url, auth=HTTPBasicAuth(USERNAME, PASSWORD))
    if r.status_code != 200:
        raise Exception("Failed to fetch sensors:", r.text)
    return r.json()

valid_statuses = {
    "ACTIVE",
    "ACTIVE_UNCONFIRMED",
    "NOT_CLAIMED",
    "NOT_CLAIMED_UNCONFIRMED"
}


sensors = get_sensors()
filtered = [s for s in sensors if s["status"] in valid_statuses]

print("Total sensors:", len(sensors))
print("Filtered sensors:", len(filtered))
print("Example sensor:", filtered[0])

Total sensors: 189
Filtered sensors: 180
Example sensor: {'sensorId': 'sensor_dev_60237_141', 'position': '42.03900255426,21.40771061182', 'comments': 'Imported Sensor.community #60237', 'type': '20004', 'description': 'Sensor.community 60237', 'status': 'NOT_CLAIMED'}


In [31]:
def parse_latlon(position):
    try:
        lat, lon = position.split(",")
        return float(lat), float(lon)
    except:
        return None, None

In [32]:
def save_week_csv(filepath, data):
    with open(filepath, "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["timestamp", "sensorId", "lat", "lon", "type", "value"])

        for entry in data:
            lat, lon = parse_latlon(entry.get("position", ""))

            writer.writerow([
                entry["stamp"],
                entry["sensorId"],
                lat,
                lon,
                entry["type"],
                entry["value"]
            ])

    print("Saved:", filepath)


In [37]:
request_count = 0
start_time_global = time.time()
def fetch_raw(sensor_id, start, end):
    global request_count

    url = f"{BASE_URL}/dataRaw"
    params = {
        "sensorId": sensor_id,
        "from": start.isoformat() + "Z",
        "to": end.isoformat() + "Z",
    }

    r = requests.get(url, params=params, auth=HTTPBasicAuth(USERNAME, PASSWORD))

    request_count += 1

    elapsed = time.time() - start_time_global
    print(f"Request #{request_count} | Elapsed: {elapsed:.1f}s | Status: {r.status_code}")

    time.sleep(0.5)  

    if r.status_code == 429:
        print("Rate limited — waiting 60 sec…")
        time.sleep(60)
        return fetch_raw(sensor_id, start, end)

    if r.status_code != 200:
        print("Error:", r.status_code, r.text)
        return []

    return r.json()


In [ ]:
def download_range_for_sensor(sensor_id, start_date, end_date):
    current = start_date

    current_year = current.year
    week_counter = 1

    while current < end_date:
        week_end = min(current + timedelta(days=7), end_date)

        if current.year != current_year:
            current_year = current.year
            week_counter = 1

        print(f"{sensor_id} | {current_year} Week {week_counter}: {current.date()} → {week_end.date()}")

        # Folder structure pulse_data/2024/week_1
        folder = f"{OUT_DIR}/{current_year}/week_{week_counter}"
        os.makedirs(folder, exist_ok=True)

        data = fetch_raw(sensor_id, current, week_end)

        filename = f"{sensor_id}_{current.date()}_{week_end.date()}.csv"
        filepath = f"{folder}/{filename}"

        save_week_csv(filepath, data)

        current = week_end
        week_counter += 1


In [35]:
test_sensor = filtered[1]
sensor_id = test_sensor["sensorId"]
print("Testing sensor:", sensor_id)


Testing sensor: sensor_dev_10699_244


In [36]:
start_date = datetime(2023, 12, 1, 0, 0, 0)
end_date   = datetime(2025, 12, 1, 0, 0, 0)
download_range_for_sensor(sensor_id, start_date, end_date)


sensor_dev_10699_244 | Week 1: 2023-12-01 → 2023-12-08
Request #1 | Elapsed: 0.3s | Status: 200
Saved: pulse_data/2023/week_1/sensor_dev_10699_244_2023-12-01_2023-12-08.csv
sensor_dev_10699_244 | Week 2: 2023-12-08 → 2023-12-15
Request #2 | Elapsed: 5.5s | Status: 200
Saved: pulse_data/2023/week_2/sensor_dev_10699_244_2023-12-08_2023-12-15.csv
sensor_dev_10699_244 | Week 3: 2023-12-15 → 2023-12-22
Request #3 | Elapsed: 10.8s | Status: 200
Saved: pulse_data/2023/week_3/sensor_dev_10699_244_2023-12-15_2023-12-22.csv
sensor_dev_10699_244 | Week 4: 2023-12-22 → 2023-12-29
Request #4 | Elapsed: 16.1s | Status: 200
Saved: pulse_data/2023/week_4/sensor_dev_10699_244_2023-12-22_2023-12-29.csv
sensor_dev_10699_244 | Week 5: 2023-12-29 → 2024-01-05
Request #5 | Elapsed: 21.5s | Status: 200
Saved: pulse_data/2023/week_5/sensor_dev_10699_244_2023-12-29_2024-01-05.csv
sensor_dev_10699_244 | Week 6: 2024-01-05 → 2024-01-12
Request #6 | Elapsed: 26.8s | Status: 200
Saved: pulse_data/2024/week_6/senso

KeyboardInterrupt: 